In [ ]:
import sys, os
sys.path.append("..")
from datetime import datetime
import torch, numpy as np

from src.data import load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

# Read Historical Data

In [ ]:
if os.path.exists("../data/historical_data0.ptt"):
    print("reading data from file...")
    _data = torch.load("../data/historical_data0.ptt")

    times_ = _data['times']
    dt = float(times_.diff().mean().round())
    print(f"dt = {dt}")

    close = _data['close']
    high = _data.get('high', close)
    low = _data.get('low', close)
    open_ = _data.get('open', close)
    volume = _data['volume']

    PAIRS_ = _data['pairs']


elif os.path.exists("../data/Kraken_OHLCVT"):
    print("loading and aligning data from raw files...")
    _data, times = load_and_align_data(PAIRS, interval=5)

    times_ = torch.tensor([t.timestamp() for t in times], dtype=torch.float64)
    dt = float(times_.diff().mean().round())
    print(f"dt = {dt}")

    close = torch.tensor(get_field(_data, 'close')).T
    high = torch.tensor(get_field(_data, 'high')).T
    low = torch.tensor(get_field(_data, 'low')).T
    open_ = torch.tensor(get_field(_data, 'open')).T
    volume = torch.tensor(get_field(_data, 'volume')).T

else:
    raise FileNotFoundError("No historical data found. Please download and prepare the data as described in the README.")

# Pre-stack Training Data

Convert list-of-dicts to `(T, N)` tensors once so `BatchedLongShortEnv` can index them
directly without per-step Python overhead.

In [ ]:
# close / high / low / open / volume are already (T, N) tensors from the data loading cell.
# times_ is already (T,) float64.  Ensure consistent dtypes then sort by timestamp.
close_t  = close.to(dtype=torch.float32)    # (T, N)
high_t   = high.to(dtype=torch.float32)     # (T, N)
low_t    = low.to(dtype=torch.float32)      # (T, N)
open_t   = open_.to(dtype=torch.float32)    # (T, N)
volume_t = volume.to(dtype=torch.float32)   # (T, N)
times_t  = times_.to(dtype=torch.float64)   # (T,)

sort_idx = torch.argsort(times_t)
times_t  = times_t[sort_idx]
open_t   = open_t[sort_idx]
close_t  = close_t[sort_idx]
high_t   = high_t[sort_idx]
low_t    = low_t[sort_idx]
volume_t = volume_t[sort_idx]

T_total = close_t.shape[0]
N_assets = close_t.shape[1]
print(f"T={T_total}, N={N_assets}  |  train split: {int(T_total*0.9)} steps")

In [ ]:
from src.environment.longshort_hierarchical_leverage import (
    LongShortHierarchicalLeverageEnv, BatchedLongShortHierarchicalLeverageEnv
)

base_t = 60
tau_p = torch.tensor([base_t*30, base_t*60*4, base_t*60*24, base_t*60*24*7], dtype=torch.float32)
print("tau_p:", (tau_p / (3600 * 24)).tolist(), "[days]")

shared_kwargs = dict(
    N=N_assets,
    C0=1_000,
    tau_p=tau_p,
    bankruptcy_threshold=10.0,
    min_open_dollars=2.0,
    transaction_eps=1e-2,
    use_dollar_volume=True,
    size_buckets=(0.50, 1.00),
    close_fee=1.0,
    open_fee=1.0,
    tax_rate=0.26,
    reward_mode="log",
    val_coeff=1.0,
    roi_coeff=1.0,
    done_reward_penalty=1.0,
    max_leverage=1.0,
    maintenance_margin_ratio=None,
    dtype=torch.float32,
    device="cpu",
    eps=1e-8,
)

# Single env for validation
env = LongShortHierarchicalLeverageEnv(**shared_kwargs, save_history=False)

# Batched env for training
B = 16
bat_env = BatchedLongShortHierarchicalLeverageEnv(B=B, **shared_kwargs)

print(f"state_dim: {env.state_dim} - action_dim: {env.action_dim} - B: {B}")
print(f"actions: hold(0), long(1..{env.N}), short({env.N+1}..{2*env.N}), close({2*env.N+1}..{3*env.N})")

# Create Agent

In [ ]:
from src.agent.aac_hierarchical_model import HierarchicalModelAACAgent

M = len(tau_p)

agent = HierarchicalModelAACAgent(
    network={
        "type": "flat_per_asset_model",
        "num_assets": env.N,
        "d_asset": 4 * M + 9,
        "d_global": 10,
        "action_dim": env.action_dim,
        "hidden_dims_asset": [256],
        "d_model": 128,
        "asset_embed_dim": 128,
        "asset_embedding_mode": "add",
        # Global memory branch: this shared recurrent state is trained by latent and reward prediction losses.
        "hidden_dims_mem": [1024, 1024],
        "hidden_dims_ff": [1024, 1024],
        "d_mem": 512,
        "d_ff": 512,
        "combine_mode": "concat",
        # Heads are feedforward by default on top of global memory. Flip these if you want extra head-specific recurrence.
        "actor_recurrent": False,
        "value_recurrent": False,
        "model_recurrent": True,
        "hidden_dims_actor": [512],
        "hidden_dims_value": [512],
        "hidden_dims_model": [512],
        "activation": torch.nn.GELU(),
        "recurrent_activation": torch.tanh,
        "recurrent_type": "gru",
        "enable_ema_encoder": True,
    },
    n_assets=env.N,
    n_buckets=env.K,
    gamma=0.9999,
    vf_coef=0.5,
    ent_coef=0.05,
    model_coef=0.02,
    model_reward_coef=100.0,
    imagine_length=16,
    n_imagined_trajectories=16,
    advantage_type="gae",
    gae_lambda=0.95,
    normalize_advantages=True,
    dtype=env.dtype,
    device="cuda",
    use_ema_target=True,
    ema_tau=0.995,
)

CHECKPOINT_PATH = f"../data/agent/aac_model_global_memory_{agent.net.name}.ptm"
if os.path.exists(CHECKPOINT_PATH):
    agent.load(CHECKPOINT_PATH)
    print(f"loaded checkpoint: {CHECKPOINT_PATH}")
else:
    print(f"starting from scratch (no checkpoint at {CHECKPOINT_PATH})")

n_params = sum(p.numel() for p in agent.net.parameters())
print(f"network params: {n_params:,}")


# Main Training Loop

In [ ]:
loss, rewards = [], []

In [ ]:
T_train = int(T_total * 0.9)

_loss, _rewards = agent.train_on_historical_bat(
    bat_env,
    open_t[:T_train], close_t[:T_train], high_t[:T_train], low_t[:T_train], volume_t[:T_train], times_t[:T_train],
    n_episodes=100,
    update_interval=64,
    burn_in_updates=1,
    max_steps=100_000,
    warm_up=10_000,
    lr=1e-5,
    optim="AdamW",
    init_optimizer=False,
    max_grad_norm=None,
)
agent.save(CHECKPOINT_PATH)

loss    += _loss
rewards += _rewards

In [ ]:
rewards = rewards[0]

In [ ]:
# Loss metrics
metric_names = next((list(ep_loss[0].keys()) for ep_loss in loss if len(ep_loss) > 0), [])
if not metric_names:
    raise ValueError("No loss metrics recorded yet.")

ref_metric = metric_names[0]
t_all = np.concatenate([
    np.linspace(i, i + 1, sum(len(np.asarray(loss_dict[ref_metric]).reshape(-1)) for loss_dict in ep_loss), endpoint=False)
    for i, ep_loss in enumerate(loss)
    if len(ep_loss) > 0
])


def flatten_metric(metric_name):
    chunks = []
    for ep_loss in loss:
        if len(ep_loss) == 0:
            continue
        chunks.append(np.concatenate([
            np.asarray(loss_dict[metric_name]).reshape(-1)
            for loss_dict in ep_loss
        ]))
    return np.concatenate(chunks) if chunks else np.array([])


palette = Category10[10]
figs = []
for i, metric_name in enumerate(metric_names):
    values = flatten_metric(metric_name)
    fig = bk.figure(
        title=metric_name.replace("_", " ").title(),
        x_axis_label="Training Iteration [Epochs]",
        y_axis_label="Value",
        width=1000,
        height=260,
    )
    fig.line(t_all[:len(values)], values, line_width=2, color=palette[i % len(palette)])
    figs.append(fig)

bk.show(bk.column(*figs))

In [ ]:
# rewards[ep] is list[list[float]] — one inner list per batch env
# Aggregate: sum across all envs for each episode
rewards_total = [sum(r_i for env_r in ep for r_i in env_r) for ep in rewards]
rewards_mean  = [sum(r_i for env_r in ep for r_i in env_r) / max(1, sum(len(env_r) for env_r in ep)) for ep in rewards]

fig = bk.figure(title="Total Rewards (all envs)", x_axis_label="Training Iteration [Episodes]", y_axis_label="Total Reward", width=900, height=320)
fig.line(list(range(len(rewards_total))), rewards_total, line_width=2, legend_label="Total Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

fig = bk.figure(title="Average Step Reward (all envs)", x_axis_label="Training Iteration [Episodes]", y_axis_label="Reward", width=900, height=320)
fig.line(list(range(len(rewards_mean))), rewards_mean, line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

# Per-env average step reward per episode
n_envs_plot = min(len(rewards[0]), 10) if rewards else 0
fig_env = bk.figure(title="Per-Env Average Step Reward", x_axis_label="Training Iteration [Episodes]", y_axis_label="Avg Reward / Step", width=900, height=320)
for env_idx in range(n_envs_plot):
    env_avg_rewards = [
        sum(ep[env_idx]) / max(1, len(ep[env_idx])) if env_idx < len(ep) and len(ep[env_idx]) > 0 else 0.0
        for ep in rewards
    ]
    fig_env.line(list(range(len(env_avg_rewards))), env_avg_rewards, line_width=2, legend_label=f"Env {env_idx}", color=Category10[10][env_idx % 10])
fig_env.legend.location = "bottom_right"
bk.show(fig_env)

In [ ]:
# Training episode debug — last episode, env 0
# Rewards alone are not enough to recover the critic bootstrap used during training.
# If you have a final-state value estimate for this plotted segment, set:
# episode_bootstrap_value = float(...)
episode = -1
env_idx = 0

ep_rewards = rewards[episode][env_idx]
steps = list(range(len(ep_rewards)))

bootstrap_value = float(globals().get("episode_bootstrap_value", 0.0))
bootstrap_label = "critic bootstrap" if "episode_bootstrap_value" in globals() else "zero bootstrap"

fig = bk.figure(title=f"Episode {episode} Env {env_idx} - Rewards", x_axis_label="Step", y_axis_label="Reward", width=900, height=320)
fig.scatter(steps, ep_rewards, size=2, color=Category10[10][4], legend_label="Reward")

returns = []
gamma = agent.gamma
G = bootstrap_value
for r in reversed(ep_rewards):
    G = r + gamma * G
    returns.insert(0, G)
    
fig.line(
    steps,
    returns,
    line_width=2,
    color=Category10[10][5],
    legend_label=f"Reward-to-go ({bootstrap_label}: {bootstrap_value:.4f})",
)

hist, edges = torch.histogram(torch.tensor(ep_rewards), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2
figh = bk.figure(title="Reward Distribution", width=300, height=320)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)
fig.legend.click_policy = "hide"

bk.show(bk.row(fig, figh))


In [ ]:
# Stop here during long training if you do not want to run validation yet.
# The validation section below can use either direct policy actions or imagined rollouts.
raise

# Validate

In [ ]:
int(T_total * 0.9)

In [ ]:
T_val_start = int(T_total * 0.9)
T_val_end   = T_val_start + 10_000 # T_total - 1
USE_IMAGINATION = False


def _make_data(idx):
    return dict(
        close=close_t[idx], high=high_t[idx], low=low_t[idx],
        open=open_t[idx], volume=volume_t[idx], time=float(times_t[idx]),
    )

agent.net.reset(1)
env.save_history = False

hist_s = []
hist_r = []
hist_i = []
hist_a = []

state = env.reset(_make_data(T_val_start))
done = False
for idx in range(T_val_start + 1, T_val_end + 1):
    mask = env.valid_action_mask()
    action = agent.act(
        state.to_tensor(),
        mask=mask,
        explore=False,
        grad_enabled=False,
        use_imagination=USE_IMAGINATION,
    )
    next_state, reward, done, info_t = env.step(action, data=_make_data(idx))

    hist_s.append(state)
    hist_r.append(float(reward))
    hist_i.append(info_t)
    hist_a.append(action.tolist())   # [a_d, a_q]

    state = next_state
    if done:
        break

if len(hist_i) == 0:
    raise ValueError("Validation rollout produced no steps.")

asset_names = list(PAIRS_.keys())
ts = [datetime.fromtimestamp(item["t"]) for item in hist_i]

Vs = torch.tensor([item["V"] for item in hist_i], dtype=torch.float32)
Cs = torch.tensor([item["C"] for item in hist_i], dtype=torch.float32)
ps = torch.tensor([item["p"] for item in hist_i], dtype=torch.float32)
pos_units = torch.tensor([item["pos_units"] for item in hist_i], dtype=torch.float32)
committed = torch.tensor([item["committed"] for item in hist_i], dtype=torch.float32)
rewards_val = torch.tensor(hist_r, dtype=torch.float32)

action_ids = torch.tensor(hist_a, dtype=torch.long)               # (T, 2)  [a_d, a_q]
# action_type: 0=hold, 1=long, 2=short, 3=close
action_types = torch.tensor([item["action_type"] for item in hist_i], dtype=torch.long)
action_assets = torch.tensor([item["action_asset"] for item in hist_i], dtype=torch.long)
valid_trades = torch.tensor([float(item["valid_trade"]) for item in hist_i], dtype=torch.float32)
realized_pnl = torch.tensor([sum(item["realized_pnl"]) for item in hist_i], dtype=torch.float32)
realized_cost = torch.tensor([sum(item["realized_cost"]) for item in hist_i], dtype=torch.float32)

# Signed exposure fractions (negative = short)
portfolio_frac = pos_units * ps / Vs[:, None].clamp_min(1e-8)
cash_frac = Cs / Vs.clamp_min(1e-8)
cum_reward = rewards_val.cumsum(0)
cum_realized_pnl = realized_pnl.cumsum(0)
cum_realized_cost = realized_cost.cumsum(0)
running_peak = torch.cummax(Vs, dim=0).values
drawdown = 1.0 - Vs / running_peak.clamp_min(1e-8)
norm_prices = ps / ps[0].clamp_min(1e-8)
norm_value = Vs / Vs[0].clamp_min(1e-8)

hold_mask  = action_types == 0
long_mask  = action_types == 1
short_mask = action_types == 2
close_mask = action_types == 3
trade_mask = long_mask | short_mask | close_mask
invalid_mask = trade_mask & (valid_trades == 0)

cumulative_longs  = long_mask.to(torch.float32).cumsum(0)
cumulative_shorts = short_mask.to(torch.float32).cumsum(0)
cumulative_closes = close_mask.to(torch.float32).cumsum(0)
cumulative_invalid = invalid_mask.to(torch.float32).cumsum(0)

validation_metrics = {
    "steps": len(hist_i),
    "terminated": bool(done),
    "final_value": float(Vs[-1].item()),
    "final_cash": float(Cs[-1].item()),
    "total_return_pct": float((norm_value[-1] - 1.0).item() * 100.0),
    "max_drawdown_pct": float(drawdown.max().item() * 100.0),
    "total_reward": float(cum_reward[-1].item()),
    "mean_reward": float(rewards_val.mean().item()),
    "reward_std": float(rewards_val.std(unbiased=False).item()),
    "long_count": int(long_mask.sum().item()),
    "short_count": int(short_mask.sum().item()),
    "close_count": int(close_mask.sum().item()),
    "hold_count": int(hold_mask.sum().item()),
    "invalid_trade_count": int(invalid_mask.sum().item()),
    "valid_trade_rate_pct": float(valid_trades.mean().item() * 100.0),
    "trade_rate_pct": float(trade_mask.to(torch.float32).mean().item() * 100.0),
    "realized_pnl_total": float(cum_realized_pnl[-1].item()),
    "realized_cost_total": float(cum_realized_cost[-1].item()),
}
validation_metrics

In [ ]:
for key, value in validation_metrics.items():
    if isinstance(value, float):
        print(f"{key:>22}: {value:.6f}")
    else:
        print(f"{key:>22}: {value}")

In [ ]:
overview_skip = max(1, len(ts) // 1500)

fig_prices = bk.figure(
    title="Validation - Normalized Prices vs Portfolio",
    width=1000,
    height=360,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Normalized Value [1]",
)
for i, name in enumerate(asset_names):
    fig_prices.line(ts[::overview_skip], norm_prices[::overview_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][i % 10])
fig_prices.line(ts[::overview_skip], norm_value[::overview_skip].numpy(), line_width=3, legend_label="Portfolio", color="black")
fig_prices.legend.click_policy = "hide"

fig_value = bk.figure(
    title="Validation - Portfolio Value and Cash",
    width=1000,
    height=320,
    x_range=fig_prices.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig_value.line(ts, Vs.numpy(), line_width=2, legend_label="Portfolio Value", color=Category10[10][0])
fig_value.line(ts, Cs.numpy(), line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][1])
fig_value.legend.location = "bottom_right"

fig_drawdown = bk.figure(
    title="Validation - Drawdown",
    width=1000,
    height=280,
    x_range=fig_prices.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Drawdown [%]",
)
fig_drawdown.line(ts, (100.0 * drawdown).numpy(), line_width=2, color=Category10[10][3], legend_label="Drawdown")
fig_drawdown.legend.location = "bottom_right"

bk.show(bk.column(fig_prices, fig_value, fig_drawdown))

In [ ]:
reward_skip = max(1, len(ts) // 2000)
reward_bins = min(200, max(20, len(rewards_val) // 10))

fig_reward = bk.figure(
    title="Validation - Step Reward",
    width=1000,
    height=280,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Reward",
)
fig_reward.line(ts[::reward_skip], rewards_val[::reward_skip].numpy(), line_width=2, color=Category10[10][4], legend_label="Step Reward")
fig_reward.legend.location = "bottom_right"

fig_cum_reward = bk.figure(
    title="Validation - Cumulative Reward",
    width=1000,
    height=280,
    x_range=fig_reward.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Cumulative Reward",
)
fig_cum_reward.line(ts, cum_reward.numpy(), line_width=2, color=Category10[10][2], legend_label="Cumulative Reward")
fig_cum_reward.legend.location = "bottom_right"

hist, edges = torch.histogram(rewards_val, bins=reward_bins, density=True)
x = ((edges[:-1] + edges[1:]) / 2).numpy()

fig_hist = bk.figure(
    title="Validation - Reward Distribution",
    width=380,
    height=320,
    x_axis_label="Density",
    y_axis_label="Reward",
)
fig_hist.harea(y=x, x1=0, x2=hist.numpy(), fill_color=Category10[10][4], fill_alpha=0.35)
fig_hist.line(hist.numpy(), x, line_color=Category10[10][4], line_width=2)

fig_realized = bk.figure(
    title="Validation - Cumulative Realized PnL and Cost",
    width=800,
    height=320,
    x_range=fig_reward.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig_realized.line(ts, cum_realized_pnl.numpy(), line_width=2, color=Category10[10][0], legend_label="Cumulative Realized PnL")
fig_realized.line(ts, cum_realized_cost.numpy(), line_width=2, color=Category10[10][1], line_dash="dashed", legend_label="Cumulative Realized Cost")
fig_realized.legend.location = "bottom_right"

bk.show(bk.column(fig_reward, fig_cum_reward, bk.row(fig_hist, fig_realized)))

In [ ]:
alloc_skip = max(1, len(ts) // 1500)

fig_alloc = bk.figure(
    title="Validation - Signed Portfolio Fractions (negative = short)",
    width=1200,
    height=360,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Fraction [1]",
)
fig_alloc.line(ts[::alloc_skip], cash_frac[::alloc_skip].numpy(), line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(asset_names):
    fig_alloc.line(ts[::alloc_skip], portfolio_frac[::alloc_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][(i + 1) % 10])
fig_alloc.legend.click_policy = "hide"

fig_actions = bk.figure(
    title="Validation - Cumulative Actions",
    width=1200,
    height=280,
    x_range=fig_alloc.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Count",
)
fig_actions.line(ts, cumulative_longs.numpy(), line_width=2, color=Category10[10][2], legend_label="Longs")
fig_actions.line(ts, cumulative_shorts.numpy(), line_width=2, color=Category10[10][3], legend_label="Shorts")
fig_actions.line(ts, cumulative_closes.numpy(), line_width=2, color=Category10[10][4], legend_label="Closes")
fig_actions.line(ts, cumulative_invalid.numpy(), line_width=2, color=Category10[10][5], legend_label="Invalid")
fig_actions.legend.location = "bottom_right"

# Committed capital over time
fig_committed = bk.figure(
    title="Validation - Committed Capital per Asset",
    width=1200,
    height=280,
    x_range=fig_alloc.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
for i, name in enumerate(asset_names):
    fig_committed.line(ts[::alloc_skip], committed[::alloc_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][i % 10])
fig_committed.legend.click_policy = "hide"
fig_committed.legend.location = "bottom_right"

bk.show(bk.column(fig_alloc, fig_actions, fig_committed))